# PolyRoute — Local Test Notebook

Verifies ALL code paths locally — no model downloads, no GPU, no FLEURS download.  
Uses `unittest.mock` to replace heavy models with lightweight stubs.

| Local Cell | Kaggle Step | What it verifies |
|---|---|---|
| Cell 2 | Step 0 | datasets version detection |
| Cell 3 | Step 1 | Backend classes importable / no download |
| Cell 4 | Step 2 | Full pipeline chain (mocked) |
| Cell 5 | Step 3 | LID eval loop (mocked 5 samples) |
| Cell 6 | Step 4 | Full eval loop (mocked 5 samples) |
| Cell 7 | Step 5 | All 4 baselines importable |
| Cell 8 | Step 6 | Policy train/save/load/predict |
| Cell 9 | Step 7 | Ablation runner importable |
| Cell 10 | — | Full pytest suite |


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))
import numpy as np
from unittest.mock import MagicMock, patch

py_ver = sys.version.split()[0]
print('Python:', py_ver)
print('CWD:', os.getcwd())

from src.utils import load_config, get_language_map, UncertaintySignals, PipelineOutput
from src.lid.fusion import LIDFusion
from src.routing.policy_rules import RuleBasedPolicy, RoutingMode
from src.routing.confusion_map import ConfusionMap
from evaluation.metrics import EvaluationResults, LIDResult, ASRResult, compute_cer
print('OK: All core imports')


Python: 3.13.12
CWD: C:\Users\ekans\Desktop\Btech\Sem-4\Sp\End_Sem_Project
OK: All core imports


In [2]:
import importlib.metadata
ver = importlib.metadata.version('datasets')
major = int(ver.split('.')[0])
print('datasets version:', ver)
if major >= 4:
    print('WARN: datasets >= 4.0 -- FLEURS will fail on Kaggle; Step 0 downgrades to 2.20.0')
    try:
        from evaluation.data_loader import load_fleurs
        load_fleurs(['en_us'])
        raise AssertionError('Should have raised RuntimeError')
    except RuntimeError as e:
        msg = str(e)[:80]
        print('OK: data_loader raises RuntimeError correctly:', msg, '...')
else:
    print('OK: datasets version compatible with FLEURS')


datasets version: 4.8.4
WARN: datasets >= 4.0 -- FLEURS will fail on Kaggle; Step 0 downgrades to 2.20.0


OK: data_loader raises RuntimeError correctly: datasets 4.8 no longer supports script-based datasets like google/fleurs.
Fix: r ...


In [3]:
from src.lid.acoustic_lid import AcousticLID
from src.lid.decoder_lid import DecoderLID
from src.asr.whisper_backend import WhisperBackend
from src.asr.mms_backend import MMSBackend

ac = AcousticLID(model_id='facebook/mms-lid-4017', device='cpu')
dc = DecoderLID(model_size='large-v3', device='cpu')
wb = WhisperBackend(model_size='large-v3', device='cpu')
mb = MMSBackend(model_id='facebook/mms-1b-all', device='cpu')

assert not ac.is_loaded()
assert not dc.is_loaded()
assert not wb.is_loaded()
assert not mb.is_loaded()
print('OK: All backends instantiate without downloading')


OK: All backends instantiate without downloading


In [4]:
from src.pipeline import Pipeline
from src.utils import PipelineOutput, TranscriptResult
import numpy as np
from unittest.mock import MagicMock, patch

dummy_audio = np.zeros(16000, dtype=np.float32)

mock_acoustic = MagicMock()
mock_acoustic.predict.return_value = {'eng': 0.9, 'deu': 0.05, 'fra': 0.05}
mock_acoustic.is_loaded.return_value = True

mock_decoder = MagicMock()
mock_decoder.predict.return_value = {'en': 0.85, 'de': 0.10, 'fr': 0.05}
mock_decoder.is_loaded.return_value = True
mock_decoder.get_model.return_value = MagicMock()

mock_whisper = MagicMock()
mock_whisper.is_loaded.return_value = True
mock_whisper.transcribe.return_value = TranscriptResult('hello world', 'eng', 0.95, 'whisper')
mock_whisper.transcribe_auto.return_value = TranscriptResult('hello', 'eng', 0.95, 'whisper_auto')
mock_whisper.unload.return_value = None

mock_mms = MagicMock()
mock_mms.is_loaded.return_value = True
mock_mms.transcribe.return_value = TranscriptResult('hello world', 'eng', 0.93, 'mms')
mock_mms.unload.return_value = None

pipe = Pipeline()
pipe.acoustic_lid = mock_acoustic
pipe.decoder_lid = mock_decoder
pipe.whisper_backend = mock_whisper
pipe.mms_backend = mock_mms
pipe._models_loaded = True

with patch('src.pipeline.preprocess', return_value=[dummy_audio]):
    out = pipe.run(dummy_audio)

assert isinstance(out, PipelineOutput)
assert out.detected_language == 'eng'
assert out.routing_mode in ['A', 'B', 'C']
print('OK: Pipeline mocked -- lang=%s, mode=%s, backend=%s' % (out.detected_language, out.routing_mode, out.backend_used))


[16:06:24 routing.confusion_map INFO] Loaded 8 confusion clusters covering 21 languages


[16:06:24 routing.agent INFO] RoutingAgent initialized with 'rules' policy


[16:06:24 pipeline INFO] Routing: mode=A, candidates=['eng'], conf=0.880


[16:06:24 decoding.single INFO] Mode A: decoding 'eng' with whisper


OK: Pipeline mocked -- lang=eng, mode=A, backend=whisper


In [5]:
from evaluation.evaluate import evaluate_lid_only
from evaluation.metrics import EvaluationResults
from src.utils import get_language_map, UncertaintySignals
from src.routing.policy_rules import RoutingDecision
import numpy as np, json, pathlib
from unittest.mock import patch

FAKE_LANGS = ['en_us', 'de_de']

def make_fake_fleurs(lang_codes, split, streaming):
    return {lc: [{'audio': {'array': np.zeros(16000, dtype=np.float32), 'sampling_rate': 16000},
                   'transcription': 'test', 'raw_transcription': 'test'} for _ in range(5)]
            for lc in lang_codes}

def fake_iterate(datasets, max_per_lang=None):
    lm = get_language_map()
    for fc, samples in datasets.items():
        canon = lm.from_fleurs(fc)
        for s in (samples[:max_per_lang] if max_per_lang else samples):
            yield s['audio']['array'], 16000, fc, canon, s['transcription']

def fake_lid(audio, sr=16000):
    return {'fused_probs': {'eng': 0.80, 'deu': 0.15},
            'top_language': 'eng',
            'uncertainty': UncertaintySignals(top1_prob=0.80, gap=0.65, entropy=0.8, top3_concentration=1.0),
            'routing_decision': RoutingDecision(mode='A', candidate_languages=['eng'], confidence=0.80, reason='mock')}

pipe2 = Pipeline()
pipe2.run_lid_only = fake_lid
pipe2._models_loaded = True

with patch('evaluation.evaluate.load_fleurs', side_effect=make_fake_fleurs), \
     patch('evaluation.evaluate.iterate_fleurs', side_effect=fake_iterate):
    r = evaluate_lid_only(pipe2, lang_codes=FAKE_LANGS, split='validation', max_per_lang=5)

pathlib.Path('results').mkdir(exist_ok=True)
with open('results/lid_only_results_local_test.json', 'w') as f:
    json.dump(r.to_dict(), f)

acc = r.lid_accuracy()
n = len(r.lid_results)
print('OK: LID eval loop -- accuracy=%.3f, samples=%d' % (acc, n))


[16:06:24 routing.confusion_map INFO] Loaded 8 confusion clusters covering 21 languages


[16:06:24 routing.agent INFO] RoutingAgent initialized with 'rules' policy


LID Evaluation: 0it [00:00, ?it/s]

LID Evaluation: 10it [00:00, 68311.14it/s]

OK: LID eval loop -- accuracy=0.500, samples=10


In [6]:
from evaluation.evaluate import evaluate_full
from src.utils import PipelineOutput
from src.pipeline import Pipeline
from unittest.mock import patch

def fake_run(audio, sr=16000, apply_vad=True):
    return PipelineOutput(transcript='hello', detected_language='eng',
                         routing_mode='A', backend_used='whisper',
                         confidence=0.92, candidates_considered=1)

pipe3 = Pipeline()
pipe3.run = fake_run
pipe3._models_loaded = True

with patch('evaluation.evaluate.load_fleurs', side_effect=make_fake_fleurs), \
     patch('evaluation.evaluate.iterate_fleurs', side_effect=fake_iterate):
    r2 = evaluate_full(pipe3, lang_codes=FAKE_LANGS, split='test', max_per_lang=5,
                       save_path='results/eval_results_local_test.json')

cer = r2.mean_cer()
routing = r2.routing_distribution()
print('OK: Full eval loop -- avg_CER=%.3f, routing=%s' % (cer, routing))


[16:06:24 routing.confusion_map INFO] Loaded 8 confusion clusters covering 21 languages


[16:06:24 routing.agent INFO] RoutingAgent initialized with 'rules' policy


Full Evaluation: 0it [00:00, ?it/s]

Full Evaluation: 10it [00:00, 298.89it/s]


[16:06:24 evaluate INFO] Evaluation complete: 10 samples in 0.0s (0.00s/sample)


...

[16:06:24 evaluate INFO] Mean CER: 1.000


[16:06:24 evaluate INFO] Routing: {'A': 10}


[16:06:24 evaluate INFO] Avg Decode Calls: 1.00


[16:06:24 evaluate INFO] Results saved to results/eval_results_local_test.json


OK: Full eval loop -- avg_CER=1.000, routing={'A': 10}


In [7]:
import inspect
from evaluation.evaluate import (
    evaluate_baseline_oracle,
    evaluate_baseline_whisper_auto,
    evaluate_baseline_static_mms,
    evaluate_baseline_static_sb_whisper,
)
for fn in [evaluate_baseline_oracle, evaluate_baseline_whisper_auto,
           evaluate_baseline_static_mms, evaluate_baseline_static_sb_whisper]:
    sig = inspect.signature(fn)
    assert 'lang_codes' in sig.parameters
    assert 'max_per_lang' in sig.parameters
    print('OK:', fn.__name__, 'importable with correct signature')


OK: evaluate_baseline_oracle importable with correct signature
OK: evaluate_baseline_whisper_auto importable with correct signature
OK: evaluate_baseline_static_mms importable with correct signature
OK: evaluate_baseline_static_sb_whisper importable with correct signature


In [8]:
from src.routing.policy_learned import generate_oracle_labels, LearnedRoutingPolicy
from src.utils import UncertaintySignals
import numpy as np, pathlib

rng = np.random.default_rng(42)
N = 100

fake_fused = []
for _ in range(N):
    a = float(rng.random())
    b = float(rng.random())
    t = a + b
    fake_fused.append({'eng': a/t, 'deu': b/t})

fake_unc = [UncertaintySignals(
    top1_prob=float(rng.random()),
    gap=float(rng.random() * 0.5),
    entropy=float(rng.random() * 3),
    top3_concentration=float(rng.random())
) for _ in range(N)]

fake_langs = rng.choice(['eng', 'deu', 'fra'], size=N).tolist()

X, y = generate_oracle_labels(fake_fused, fake_unc, fake_langs)
assert X.shape == (N, 6), 'Expected (100,6) got %s' % str(X.shape)

policy = LearnedRoutingPolicy()
history = policy.train_policy(X, y, epochs=10, lr=0.01)
val_acc = history['val_acc'][-1]
print('Training done -- val_acc=%.3f' % val_acc)

pathlib.Path('models').mkdir(exist_ok=True)
policy.save('models/routing_policy_test.pt')

policy2 = LearnedRoutingPolicy()
policy2.load('models/routing_policy_test.pt')
dec = policy2.decide({'eng': 0.8, 'deu': 0.2}, fake_unc[0])
assert dec.mode in ['A', 'B', 'C']
print('OK: Policy train/save/load/predict -- mode=%s' % dec.mode)


[16:06:31 routing.policy_learned INFO] Epoch 10/10: train_loss=1.0863, val_loss=1.1011, val_acc=0.467


[16:06:31 routing.policy_learned INFO] Learned policy saved to models/routing_policy_test.pt


[16:06:31 routing.policy_learned INFO] Learned policy loaded from models/routing_policy_test.pt


Training done -- val_acc=0.467
OK: Policy train/save/load/predict -- mode=B


In [9]:
import inspect
from evaluation.ablation import (
    run_ablation_a1, run_ablation_a2, run_ablation_a3, run_ablation_a4,
    run_ablation_a5, run_ablation_a6, run_ablation_a7, run_ablation_a8,
    run_all_ablations,
)
for fn in [run_ablation_a1, run_ablation_a2, run_ablation_a3, run_ablation_a4,
           run_ablation_a5, run_ablation_a6, run_ablation_a7, run_ablation_a8,
           run_all_ablations]:
    assert callable(fn)
    assert 'max_per_lang' in inspect.signature(fn).parameters
    print('OK:', fn.__name__, 'importable')


OK: run_ablation_a1 importable
OK: run_ablation_a2 importable
OK: run_ablation_a3 importable
OK: run_ablation_a4 importable
OK: run_ablation_a5 importable
OK: run_ablation_a6 importable
OK: run_ablation_a7 importable
OK: run_ablation_a8 importable
OK: run_all_ablations importable


In [10]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, '-m', 'pytest', 'tests/', '-v', '--tb=short'],
    capture_output=True, text=True
)
out = result.stdout
print(out[-3000:] if len(out) > 3000 else out)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
    raise AssertionError('Unit tests FAILED')
print('OK: All unit tests passed')


============================= test session starts =============================
platform win32 -- Python 3.13.12, pytest-9.0.2, pluggy-1.6.0 -- C:\Users\ekans\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\ekans\Desktop\Btech\Sem-4\Sp\End_Sem_Project
plugins: anyio-4.6.2.post1
collecting ... collected 20 items

tests/test_decoding.py::TestSingleDecode::test_decode_single PASSED      [  5%]
tests/test_decoding.py::TestMultiHypothesis::test_reranking PASSED       [ 10%]
tests/test_decoding.py::TestFallbackDecode::test_tiered_fallback PASSED  [ 15%]
tests/test_lid.py::TestLIDFusion::test_fusion_both_signals PASSED        [ 20%]
tests/test_lid.py::TestLIDFusion::test_fusion_disagreement PASSED        [ 25%]
tests/test_lid.py::TestLIDFusion::test_fusion_single_signal PASSED       [ 30%]
tests/test_lid.py::TestLIDFusion::test_uncertainty_confident PASSED      [ 35%]
tests/test_lid.py::TestLIDFusion::

In [11]:
print('=' * 55)
print('LOCAL TEST SUMMARY')
print('=' * 55)
checks = [
    'Core imports',
    'datasets version detection + error guard',
    'Backend classes instantiate (no downloads)',
    'Full pipeline chain (mocked)',
    'LID eval loop (mocked 5 samples + JSON save)',
    'Full eval loop (mocked 5 samples)',
    'All 4 baselines importable + signature check',
    'Learned policy train/save/load/predict',
    'All A1-A8 ablations importable',
    'pytest suite (20 tests)',
]
for c in checks:
    print('  OK:', c)
print('=' * 55)
print('Kaggle Notebook is ready to run.')


LOCAL TEST SUMMARY
  OK: Core imports
  OK: datasets version detection + error guard
  OK: Backend classes instantiate (no downloads)
  OK: Full pipeline chain (mocked)
  OK: LID eval loop (mocked 5 samples + JSON save)
  OK: Full eval loop (mocked 5 samples)
  OK: All 4 baselines importable + signature check
  OK: Learned policy train/save/load/predict
  OK: All A1-A8 ablations importable
  OK: pytest suite (20 tests)
Kaggle Notebook is ready to run.
